# TwinHeat · train one FNO ensemble member (M2.2)

Kaggle template: clones the repo **at a tag**, checks the GPU, and runs `models/train_ensemble.py`
for one member and one stage. Settings: **Accelerator GPU T4 x2, Internet on**; attach the dataset v1
Kaggle Dataset (N2.2 release) and, if the repo is private, a `GITHUB_TOKEN` secret.

1. Set the cell below (member, seed, stage, epochs, tag). Seeds and epochs for reported runs come from
   `experiments/configs/frozen.yaml` (CLAUDE.md §4).
2. Run all. Checkpoints are written every epoch to `/kaggle/working/member<M>_<stage>/`.
3. **Save Version** (Save & Run All, or Quick Save after the run) so the output survives the session.
   A session that ran out of time is resumed by attaching that version's output and setting `RESUME_FROM`.
4. Member 1 first: its `summary.json` gives the GPU-hours that set the schedule for members 2–5.
5. Download `summary.json`, `log.csv`, `best.pt` and `registry_member<M>.json`; the registry entries are
   merged into `models/registry.json` in a PR.

In [ ]:
TAG = None          # git tag to train from, e.g. the tag that froze the training config
MEMBER = 1          # 1..5
SEED = None         # this member's seed (frozen.yaml)
STAGE = "pre128"    # "pre128" first, then "ft256"
EPOCHS = None       # frozen.yaml
DATA = "/kaggle/input/twinheat-dataset-v1"  # dataset v1: one folder per split
INIT = None         # ft256 only: this member's pre128 best.pt (an attached input)
RESUME_FROM = None  # optional: an earlier version's /kaggle/working/member<M>_<stage>
WORKERS = 2         # DataLoader workers (Kaggle gives 4 CPUs)

for name, v in {"TAG": TAG, "SEED": SEED, "EPOCHS": EPOCHS}.items():
    assert v is not None, f"set {name}"
assert STAGE in ("pre128", "ft256") and (STAGE == "pre128" or INIT), "ft256 needs INIT"
OUT = f"/kaggle/working/member{MEMBER}_{STAGE}"

In [ ]:
import subprocess

try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
    url = f"https://{token}@github.com/Akxsh65/twinheat.git"
except Exception:
    url = "https://github.com/Akxsh65/twinheat.git"
REPO = "/kaggle/working/twinheat"
subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", TAG, url, REPO], check=True)
%cd /kaggle/working/twinheat
!git log --oneline -1

In [ ]:
!pip install -q pydantic==2.14.1 jsonschema==4.26.0 pyyaml==6.0.3
!python scripts/check_gpu.py | tail -1
!test -f models/norm_stats.json && echo "norm_stats.json present" || echo "MISSING norm_stats"

In [ ]:
import os
import shutil

if RESUME_FROM:
    for sub in ("ckpt", "backup"):
        src = os.path.join(RESUME_FROM, sub)
        if os.path.isdir(src):
            shutil.copytree(src, os.path.join(OUT, sub), dirs_exist_ok=True)
    print("restored", os.listdir(os.path.join(OUT, "ckpt")))

In [ ]:
cmd = (f"python -m models.train_ensemble --member {MEMBER} --seed {SEED} --stage {STAGE} "
       f"--epochs {EPOCHS} --data {DATA} --ckpt_dir {OUT}/ckpt --backup_dir {OUT}/backup "
       f"--registry /kaggle/working/registry_member{MEMBER}.json --workers {WORKERS}")
if INIT:
    cmd += f" --init {INIT}"
print(cmd)
!{cmd} 2>&1 | tee {OUT}/train_log.txt

In [ ]:
print(open(f"{OUT}/ckpt/summary.json").read())
!tail -5 {OUT}/ckpt/log.csv